# Calibration analysis (Black / Heston / SVCJ) — thesis figures & tables

This notebook turns `calibration_results.xlsx` into the figures and tables used in the thesis calibration-results section.

**Design choices**
- **Snapshot is the sampling unit**. Where we report confidence intervals for average errors, we bootstrap **over snapshots** (not over individual quotes).
- Errors are measured in **coin premium units** (inverse option numeraire).
- We report both **price-space errors** (RMSE/MAE) and **microstructure-aware** diagnostics (within-spread fractions, error/spread, etc.).

> If you moved the Excel file, update `DATA_PATH` in the next cell.


In [1]:
from pathlib import Path
import sys

from IPython.display import display

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib import chapter3_analysis as analysis
from _lib.common import ensure_existing_path, locate_notebook_paths

RNG = analysis.initialize_notebook_defaults()
PATHS = locate_notebook_paths(NOTEBOOK_DIR)
DATA_PATH = "calibration_results_reg_50.xlsx"
MODEL_SPECS = analysis.MODEL_SPECS
COLORS = analysis.COLORS
FIGDIMS = analysis.FIGDIMS
DATA_FILE = ensure_existing_path(Path(DATA_PATH), search_dirs=[PATHS.excel_dir, PATHS.project_root, PATHS.notebook_dir])


/opt/miniconda3/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [2]:
state = analysis.build_analysis_state(DATA_FILE, rng=RNG)

black_params = state.black_params
heston_params = state.heston_params
svcj_params = state.svcj_params
train_data = state.train_data
test_data = state.test_data

print("Loaded from:", DATA_FILE)
print(" - black_params:", black_params.shape)
print(" - heston_params:", heston_params.shape)
print(" - svcj_params:", svcj_params.shape)
print(" - train_data:", train_data.shape)
print(" - test_data :", test_data.shape)

display(black_params.head(3))


Loaded from: /Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/excel files/calibration_results_reg_50.xlsx
 - black_params: (776, 16)
 - heston_params: (776, 20)
 - svcj_params: (776, 25)
 - train_data: (248526, 34)
 - test_data : (107026, 34)


/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:576: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:582: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:576: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current b

,timestamp,currency,success,message,nfev,rmse_fit,mae_fit,rmse_train,mae_train,rmse_test,mae_test,n_options_total,n_train,n_test,random_seed,sigma
0,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005825,0.003963,0.005825,0.003963,0.004958,0.003629,398,278,120,123,0.433832
1,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,4,0.005975,0.004278,0.005975,0.004278,0.006321,0.003966,392,274,118,124,0.431057
2,2025-12-31 09:18:28+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005681,0.003941,0.005681,0.003941,0.004456,0.003474,391,273,118,125,0.445089


## 1) Build snapshot-level “results” tables (metrics + convergence + parameters)

We consolidate the three model-specific parameter sheets into a common long format:

- One row per *(snapshot, currency, model)*  
- With: train/test RMSE & MAE, success flag, solver message, `nfev`, and calibrated parameters.


In [3]:
results_long = state.results_long
results_ok = state.results_ok

display(results_long.head(6))
print("Currencies:", results_long["currency"].unique())


,timestamp,currency,success,message,nfev,rmse_fit,mae_fit,rmse_train,mae_train,rmse_test,mae_test,n_options_total,n_train,n_test,random_seed,sigma,model,kappa,theta,sigma_v,rho,v0,lam,ell_y,sigma_y,ell_v,rho_j
0,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005825,0.003963,0.005825,0.003963,0.004958,0.003629,398,278,120,123,0.433832,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,30,0.002431,0.001201,0.002431,0.001201,0.001521,0.001009,398,278,120,123,NaN,Heston,5.731788,0.267392,1.755150,-0.214405,0.146301,NaN,NaN,NaN,NaN,NaN
2,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,55,0.002106,0.000700,0.002106,0.000700,0.001286,0.000503,398,278,120,123,NaN,SVCJ,3.438955,0.095675,0.514601,-0.202938,0.118918,1.184894,-0.064701,0.204992,0.407451,-0.073967
3,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,4,0.005975,0.004278,0.005975,0.004278,0.006321,0.003966,392,274,118,124,0.431057,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,10,0.002590,0.001375,0.002590,0.001375,0.003187,0.001261,392,274,118,124,NaN,Heston,6.161634,0.263701,1.806555,-0.198068,0.142629,NaN,NaN,NaN,NaN,NaN
5,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,6,0.002393,0.000873,0.002393,0.000873,0.002774,0.000776,392,274,118,124,NaN,SVCJ,3.741507,0.097972,0.497171,-0.200683,0.115044,1.217133,-0.074589,0.205029,0.404914,-0.048921


Currencies: ['BTC' 'ETH']


## 2) Option-level metrics derived from `train_data` and `test_data`

The parameter sheets already contain RMSE/MAE train/test, but option-level data lets us compute
additional diagnostics (spread-relative errors, within-spread fractions, bucket analyses, etc.).


In [4]:
opt_metrics = state.opt_metrics

display(opt_metrics.head(6))
print("Option-derived snapshot metrics:", opt_metrics.shape)


,currency,snapshot_ts,n,mse,mae,within_spread,within_half_spread,abs_err_over_spread,smape,mean_price,rmse,rmse_over_mean_price,model,split
0,BTC,2025-12-30 17:31:15+00:00,120,0.000025,0.003629,0.225000,0.150000,3.140859,0.248498,0.083687,0.004958,0.059246,Black,test
1,BTC,2025-12-30 17:31:15+00:00,278,0.000034,0.003963,0.291367,0.233813,2.927041,0.215789,0.121861,0.005825,0.047797,Black,train
2,BTC,2025-12-30 17:31:15+00:00,120,0.000002,0.001009,0.658333,0.408333,1.157900,0.151182,0.083687,0.001521,0.018175,Heston,test
3,BTC,2025-12-30 17:31:15+00:00,278,0.000006,0.001201,0.744604,0.500000,0.887746,0.111333,0.121861,0.002431,0.019947,Heston,train
4,BTC,2025-12-30 17:31:15+00:00,120,0.000002,0.000503,0.925000,0.808333,0.329393,0.028150,0.083687,0.001286,0.015363,SVCJ,test
5,BTC,2025-12-30 17:31:15+00:00,278,0.000004,0.000700,0.960432,0.874101,0.247692,0.019607,0.121861,0.002106,0.017280,SVCJ,train


Option-derived snapshot metrics: (4652, 14)


## 3) Bootstrap helpers (snapshot-level)

We treat each snapshot as one observation. For a snapshot-level series \(m_t\), we report:
- mean + 95% bootstrap CI for the mean (percentile bootstrap),
- median, quartiles, standard deviation, and sample size.


In [5]:
def bootstrap_mean_ci(values, n_boot=3000, alpha=0.05, rng=RNG):
    return analysis.bootstrap_mean_ci(values, n_boot=n_boot, alpha=alpha, rng=rng)


def summarize_snapshot_series(values, n_boot=3000):
    return analysis.summarize_snapshot_series(values, n_boot=n_boot, rng=RNG)


## 4) Plot helpers (time-series and boxplots)

We keep the same **2×2 subplot layout** you already use:

1) RMSE (all models)  
2) MAE (all models)  
3) RMSE (Heston vs SVCJ)  
4) MAE (Heston vs SVCJ)


In [6]:
add_line = analysis.add_line
add_box = analysis.add_box
add_subplot_legend = analysis.add_subplot_legend
plot_error_timeseries = analysis.plot_error_timeseries
plot_error_boxplots = analysis.plot_error_boxplots


## 5) Summary tables (errors + CI bands + incremental gains)

This produces:
- per-model summary stats (mean + 95% CI, median, quartiles, etc.)
- incremental gains and win rates for:
  - Heston vs Black
  - SVCJ vs Heston


In [7]:
def error_summary_table(results_long_df, currency, split="test", n_boot=3000):
    return analysis.error_summary_table(results_long_df, currency, split=split, n_boot=n_boot, rng=RNG)


## 6) Convergence diagnostics (success, termination messages, nfev)

We summarize by *(currency, model)*:
- number of snapshots
- success rate
- `nfev` distribution (median / p90 / max)
- how often the solver hit the max evaluation cap (detected from termination message)


In [8]:
convergence_table = analysis.convergence_table

display(convergence_table(results_long))


,currency,model,n_snapshots,success_rate,nfev_median,nfev_mean,nfev_p90,nfev_max,hit_cap_rate,top_message_1,top_message_2,top_message_3
0,BTC,Black,388,1.000000,4.0,4.134021,5.0,7,0.000000,`ftol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...,`xtol` termination condition is satisfied.
1,BTC,Heston,388,1.000000,10.0,11.159794,17.3,32,0.000000,`ftol` termination condition is satisfied.,NaN,NaN
2,BTC,SVCJ,388,0.994845,9.0,13.920103,18.3,250,0.005155,`ftol` termination condition is satisfied.,`xtol` termination condition is satisfied.,The maximum number of function evaluations is ...
3,ETH,Black,388,1.000000,4.0,4.149485,5.0,6,0.000000,`ftol` termination condition is satisfied.,NaN,NaN
4,ETH,Heston,388,1.000000,8.0,8.994845,13.0,53,0.000000,`ftol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...,NaN
5,ETH,SVCJ,388,1.000000,9.0,13.453608,20.3,238,0.000000,`ftol` termination condition is satisfied.,`xtol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...


## 7) Spread-relative diagnostics (test and train)

Using option-level quotes, we compute per-snapshot:
- fraction of quotes priced within **half-spread** and within the **full spread**
- mean \(|error|/spread\)
- sMAPE and RMSE normalized by mean market premium

We plot these over time and summarize them with bootstrap CIs.


In [9]:
spread_metric_timeseries = analysis.spread_metric_timeseries


def spread_metric_summary_table(opt_metrics_df, currency, split="test", n_boot=3000):
    return analysis.spread_metric_summary_table(opt_metrics_df, currency, split=split, n_boot=n_boot, rng=RNG)


## 8) Error breakdowns by moneyness and maturity (test set)

We report **MAE** broken down by:
- absolute log-moneyness \(|\log(K/F)|\) buckets
- maturity buckets (based on time-to-maturity in years)

Bucket metrics are computed **within each snapshot**, then averaged across snapshots (equal-weighted over time).


In [10]:
MONEY_BINS = analysis.MONEY_BINS
MONEY_LABELS = analysis.MONEY_LABELS
T_BINS = analysis.T_BINS
T_LABELS = analysis.T_LABELS
bucket_mae_by_snapshot = analysis.bucket_mae_by_snapshot
bucket_all = state.bucket_all

display(bucket_all.head(6))


,snapshot_ts,bucket,mae,model,split,bucket_type,currency
0,2025-12-30 17:31:15+00:00,|log(K/F)|≤0.05,0.003093,Black,test,moneyness,BTC
1,2025-12-30 17:31:15+00:00,0.05–0.15,0.003357,Black,test,moneyness,BTC
2,2025-12-30 17:31:15+00:00,0.15–0.30,0.004060,Black,test,moneyness,BTC
3,2025-12-30 17:31:15+00:00,>0.30,0.004229,Black,test,moneyness,BTC
4,2025-12-30 21:17:51+00:00,|log(K/F)|≤0.05,0.003925,Black,test,moneyness,BTC
5,2025-12-30 21:17:51+00:00,0.05–0.15,0.003334,Black,test,moneyness,BTC


In [11]:
def bucket_summary_table(bucket_df, currency, bucket_type, n_boot=2000):
    return analysis.bucket_summary_table(bucket_df, currency, bucket_type, n_boot=n_boot, rng=RNG)


plot_bucket_bars = analysis.plot_bucket_bars


## 9) Parameter stability and bound-pressure diagnostics

We provide:
- time-series plots for key parameters (Heston and SVCJ),
- distribution boxplots,
- “near-bound” rates using the calibration bounds (in natural parameter space),
- and the Heston/SVCJ **Feller ratio** \(\sigma_v^2/(2\kappa\theta)\) as a constraint-pressure proxy.


In [12]:
RHO_LB = analysis.RHO_LB
RHO_UB = analysis.RHO_UB
BOUNDS = analysis.BOUNDS
EPS = analysis.EPS
add_feller_ratio = analysis.add_feller_ratio
near_bound_rates = analysis.near_bound_rates
results_ok = add_feller_ratio(results_ok)


In [13]:
plot_param_timeseries = analysis.plot_param_timeseries
plot_param_boxplots = analysis.plot_param_boxplots


## 10) A single “report runner” per currency

To keep the notebook readable, we wrap the repeated steps into one function that:
- prints key counts,
- displays error time series + boxplots (train & test),
- outputs error summary tables (train & test),
- outputs spread-relative diagnostics (train & test),
- outputs bucket plots (test),
- outputs convergence diagnostics (already global),
- outputs parameter stability and near-bound tables.


In [14]:
def run_currency_report(currency, n_boot=3000):
    return analysis.run_currency_report(state, currency, n_boot=n_boot)


RUN_REPORTS = True
N_BOOT = 3000

if RUN_REPORTS:
    run_currency_report("BTC", n_boot=N_BOOT)
    run_currency_report("ETH", n_boot=N_BOOT)
else:
    print("RUN_REPORTS=False. Set RUN_REPORTS=True to generate the full BTC/ETH report outputs.")


REPORT — BTC
Snapshots: 388
Success rates:


,success_rate
model,
Black,1.000000
Heston,1.000000
SVCJ,0.994845


Summary table — BTC / train


,currency,split,metric,item,n,mean,ci_95,median,q25,q75,std,min,max,win_rate
0,BTC,train,MAE,Black,388,0.003585,"[0.0035065, 0.00366983]",0.003526,0.003016,0.004043,0.000830,0.001887,0.011706,NaN
1,BTC,train,MAE,GAIN: Black→Heston (%),388,0.584756,"[0.569178, 0.600463]",0.569664,0.470369,0.682954,0.158009,0.088749,0.893421,1.000000
2,BTC,train,MAE,GAIN: Black→Heston (abs),388,0.002154,"[0.00206191, 0.0022408]",0.001893,0.001528,0.002793,0.000931,0.000244,0.008579,1.000000
3,BTC,train,MAE,GAIN: Heston→SVCJ (%),386,0.306924,"[0.293089, 0.320358]",0.308942,0.226125,0.388694,0.134949,-0.186644,0.678803,0.981959
4,BTC,train,MAE,GAIN: Heston→SVCJ (abs),386,0.000450,"[0.000424793, 0.000475761]",0.000454,0.000267,0.000601,0.000261,-0.000296,0.001297,0.981959
5,BTC,train,MAE,Heston,388,0.001431,"[0.00137827, 0.00148243]",0.001460,0.001103,0.001717,0.000525,0.000456,0.003533,NaN
6,BTC,train,MAE,SVCJ,386,0.000975,"[0.000935995, 0.00101501]",0.000940,0.000713,0.001187,0.000394,0.000249,0.003121,NaN
7,BTC,train,RMSE,Black,388,0.005268,"[0.00515292, 0.00539136]",0.005155,0.004413,0.005968,0.001191,0.002757,0.016334,NaN
8,BTC,train,RMSE,GAIN: Black→Heston (%),388,0.495887,"[0.473962, 0.517327]",0.458440,0.344752,0.603368,0.215552,-0.009710,0.907303,0.997423
9,BTC,train,RMSE,GAIN: Black→Heston (abs),388,0.002699,"[0.00254052, 0.00285732]",0.002191,0.001558,0.003395,0.001589,-0.000038,0.010782,0.997423


Summary table — BTC / test


...

,currency,bucket_type,bucket,model,n,mean,ci_95,median,q25,q75
2,ETH,maturity,1m–3m,Black,388,0.003560,"[0.00345692, 0.00366525]",0.003379,0.002852,0.004031
6,ETH,maturity,1m–3m,Heston,388,0.002162,"[0.00203854, 0.00228579]",0.001990,0.001300,0.002794
10,ETH,maturity,1m–3m,SVCJ,388,0.001669,"[0.00156168, 0.00177934]",0.001388,0.000867,0.002233
1,ETH,maturity,1w–1m,Black,388,0.002823,"[0.00271633, 0.00293112]",0.002701,0.002123,0.003501
5,ETH,maturity,1w–1m,Heston,388,0.001512,"[0.00143466, 0.00159294]",0.001404,0.000858,0.001971
9,ETH,maturity,1w–1m,SVCJ,388,0.001179,"[0.00110497, 0.0012573]",0.000948,0.000632,0.001555
3,ETH,maturity,>3m,Black,388,0.006405,"[0.00606566, 0.00680758]",0.005516,0.004286,0.007499
7,ETH,maturity,>3m,Heston,388,0.003060,"[0.0029148, 0.00321638]",0.002919,0.001984,0.003955
11,ETH,maturity,>3m,SVCJ,388,0.002314,"[0.00218532, 0.00244792]",0.002115,0.001260,0.003014
0,ETH,maturity,≤1w,Black,385,0.002362,"[0.00222965, 0.00249705]",0.002056,0.001414,0.002923


Parameter stability — Heston


,model,param,lb,ub,near_lb_rate,near_ub_rate,min,q25,median,q75,max
0,Heston,kappa,0.000100,50.000000,0.000000,0.131443,2.872107,13.215133,20.058626,34.358028,50.000000
1,Heston,rho,-0.999909,0.999909,0.000000,0.000000,-0.545393,-0.251988,-0.217671,-0.177371,-0.077707
2,Heston,sigma_v,0.000100,10.000000,0.000000,0.000000,1.901767,3.716300,4.499073,5.799921,7.758477
3,Heston,theta,0.000001,5.000000,0.000000,0.000000,0.411111,0.464527,0.500518,0.534926,0.629067
4,Heston,v0,0.000001,5.000000,0.007732,0.000000,0.067867,0.287616,0.479978,0.598517,2.329064


Parameter stability — SVCJ


,model,param,lb,ub,near_lb_rate,near_ub_rate,min,q25,median,q75,max
0,SVCJ,ell_v,0.000001,10.000000,0.368557,0.041237,0.051164,0.171709,0.395922,1.093065,10.000000
1,SVCJ,ell_y,-5.000000,5.000000,0.000000,0.000000,-0.508394,-0.207738,-0.134927,-0.039754,0.504135
2,SVCJ,kappa,0.000100,50.000000,0.000000,0.131443,1.868586,6.909898,14.937196,32.332955,50.000000
3,SVCJ,lam,0.000001,10.000000,0.000000,0.000000,0.330035,1.326137,2.084353,2.854084,8.820435
4,SVCJ,rho,-0.999909,0.999909,0.115979,0.000000,-0.999610,-0.060702,0.125589,0.260056,0.653393
5,SVCJ,rho_j,-0.999909,0.999909,0.533505,0.000000,-0.999908,-0.999000,-0.998998,-0.010751,0.370610
6,SVCJ,sigma_v,0.000100,10.000000,0.069588,0.000000,0.101707,1.468666,2.255625,3.817597,6.071553
7,SVCJ,sigma_y,0.000100,5.000000,0.886598,0.000000,0.000109,0.000278,0.000278,0.001008,0.243422
8,SVCJ,theta,0.000001,5.000000,0.115979,0.000000,0.053871,0.191510,0.253518,0.307244,0.407863
9,SVCJ,v0,0.000001,5.000000,0.010309,0.000000,0.050186,0.244578,0.378619,0.475594,1.985584


## 11) Export thesis artifacts (tables + figures)

This cell saves:
- tables into `./tables/`
- figures into `./figs/` as HTML (always) and PNG (if Kaleido is available)

Set `EXPORT = True` to activate.


In [15]:
EXPORT = False

OUT_TABLES = Path("tables")
OUT_FIGS = Path("figs")


def _safe_write_image(fig, path_png):
    try:
        fig.write_image(path_png, scale=2)
        return True
    except Exception as exc:
        print(f"[warn] Could not write PNG (needs kaleido): {path_png}\n  {exc}")
        return False


if EXPORT:
    OUT_TABLES.mkdir(parents=True, exist_ok=True)
    OUT_FIGS.mkdir(parents=True, exist_ok=True)

    conv = convergence_table(results_long)
    conv.to_csv(OUT_TABLES / "convergence_table.csv", index=False)

    for currency in results_long["currency"].unique():
        for split in ["train", "test"]:
            tbl = error_summary_table(results_long, currency, split=split)
            tbl.to_csv(OUT_TABLES / f"{currency.lower()}_{split}_error_summary.csv", index=False)

            tbl_sp = spread_metric_summary_table(opt_metrics, currency, split=split)
            tbl_sp.to_csv(OUT_TABLES / f"{currency.lower()}_{split}_spread_summary.csv", index=False)

            fig_ts = plot_error_timeseries(results_long, currency, split=split)
            fig_ts.write_html(OUT_FIGS / f"{currency.lower()}_{split}_errors_timeseries.html")
            _safe_write_image(fig_ts, OUT_FIGS / f"{currency.lower()}_{split}_errors_timeseries.png")

            fig_box = plot_error_boxplots(results_long, currency, split=split)
            fig_box.write_html(OUT_FIGS / f"{currency.lower()}_{split}_errors_boxplots.html")
            _safe_write_image(fig_box, OUT_FIGS / f"{currency.lower()}_{split}_errors_boxplots.png")

            fig_sp = spread_metric_timeseries(opt_metrics, currency, split=split)
            fig_sp.write_html(OUT_FIGS / f"{currency.lower()}_{split}_spread_timeseries.html")
            _safe_write_image(fig_sp, OUT_FIGS / f"{currency.lower()}_{split}_spread_timeseries.png")

        b1 = bucket_summary_table(bucket_all, currency, "moneyness")
        b2 = bucket_summary_table(bucket_all, currency, "maturity")
        b1.to_csv(OUT_TABLES / f"{currency.lower()}_test_bucket_moneyness.csv", index=False)
        b2.to_csv(OUT_TABLES / f"{currency.lower()}_test_bucket_maturity.csv", index=False)

        fig_bm = plot_bucket_bars(bucket_all, currency, "moneyness")
        fig_bt = plot_bucket_bars(bucket_all, currency, "maturity")
        fig_bm.write_html(OUT_FIGS / f"{currency.lower()}_test_bucket_moneyness.html")
        fig_bt.write_html(OUT_FIGS / f"{currency.lower()}_test_bucket_maturity.html")
        _safe_write_image(fig_bm, OUT_FIGS / f"{currency.lower()}_test_bucket_moneyness.png")
        _safe_write_image(fig_bt, OUT_FIGS / f"{currency.lower()}_test_bucket_maturity.png")

        nb_hes = near_bound_rates(results_long[results_long["currency"] == currency], "Heston")
        nb_svcj = near_bound_rates(results_long[results_long["currency"] == currency], "SVCJ")
        nb_hes.to_csv(OUT_TABLES / f"{currency.lower()}_heston_near_bound_rates.csv", index=False)
        nb_svcj.to_csv(OUT_TABLES / f"{currency.lower()}_svcj_near_bound_rates.csv", index=False)

    print("Export complete.")
else:
    print("EXPORT=False (no files written). Set EXPORT=True to save tables/figures.")


EXPORT=False (no files written). Set EXPORT=True to save tables/figures.
